# NB06 Supplementary — EUR-Lex / Cellar Debugging Lab
**Scientific AI Builder · Regulatory Agent · 2026-10-19**

Purpose: reproduce the real troubleshooting journey **without modifying NB06 v3**. The distinction is **HTTP access ≠ metadata availability ≠ validated consolidation dates ≠ current legal applicability**. Run from the repository root. All live checks need internet but **not an OpenAI API key**.

## Incident timeline

| Stage | Observation | Lesson |
|---|---|---|
| EUR-Lex HTML | HTTP 202, zero-byte body | No HTML to parse; cause of 202 unconfirmed |
| Cellar resource RDF | HTTP 200, 46,497 bytes, valid RDF | Endpoint reachable; returned graph did not expose a usable consolidation link with our assumptions |
| First SPARQL | HTTP 200, 0 relation/index results | Working endpoint does not guarantee correct predicate/query |
| SPARQL diagnostic | HTTP 200, 19 CELEX rows | Search `cdm:resource_legal_id_celex`, `FILTER(CONTAINS(STR(?celex), "2019R1793"))` |
| Final NB06 v3 | `metadata_found`, 2026-06-30 equals snapshot | **Observed consolidation candidate only**, not article-level applicability proof |

This supplementary notebook documents failed hypotheses and the successful minimal query; it deliberately does not assert that a 0-result query proves no consolidation.

In [ ]:
# Resolve paths from the repository root, including nested Jupyter kernel CWDs.
import os as _repo_os
from pathlib import Path as _RepoPath
_repo_start = _RepoPath(_repo_os.environ.get("REGULATORY_AGENT_ROOT", _RepoPath.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_repo_start, *_repo_start.parents)
                  if (p / "data/benchmark/benchmark_questions_v1.json").is_file()
                  and (p / "notebooks/build04/NB09_v3.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REGULATORY_AGENT_ROOT to the Regulatory_agent checkout.")
_repo_os.chdir(REPO_ROOT)

from pathlib import Path
from datetime import datetime, timezone
import json, re
import requests
import xml.etree.ElementTree as ET

DATA_DIR=Path("data")
assert DATA_DIR.is_dir(), "Run from repository root"
DIAG_DIR=DATA_DIR/"nb06_supplementary"
DIAG_DIR.mkdir(exist_ok=True)
CELEX="32019R1793"
SNAPSHOT="2026-06-30"
ENDPOINT="https://publications.europa.eu/webapi/rdf/sparql"
print("Ready. No OpenAI API is used in this notebook.")

## 1. Failure mode: EUR-Lex HTML (optional live request)
This reproduces an **observed** HTTP 202/empty-body scenario; new requests might behave differently. Record metadata only.

In [ ]:
RUN_LIVE_HTML=False  # opt in; not necessary for the fix
if RUN_LIVE_HTML:
    url=f"https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:{CELEX}"
    try:
        r=requests.get(url,timeout=20)
        print({"status":r.status_code,"bytes":len(r.content),"content_type":r.headers.get("Content-Type"),"final_url":r.url})
    except requests.RequestException as exc:
        print("UNVERIFIED — request error:",str(exc))
else:
    print("Historical observed result: HTTP 202, Content-Type text/html, body length 0")

## 2. Cellar RDF: successful connectivity, unsuccessful consolidation extraction
The RDF returned by the base act is **not** guaranteed to include inbound consolidated-version links. This is a discovery demonstration, not a recommended currentness query.

In [ ]:
RUN_LIVE_RDF=False  # optional
if RUN_LIVE_RDF:
    url=f"https://publications.europa.eu/resource/celex/{CELEX}"
    try:
        r=requests.get(url,headers={"Accept":"application/rdf+xml;notice=non-inferred"},timeout=30)
        print("Status:",r.status_code,"bytes:",len(r.content),"type:",r.headers.get("Content-Type"))
        if r.ok and r.content:
            root=ET.fromstring(r.content)
            names=sorted({e.tag.split('}')[-1] for e in root.iter() if 'consolidat' in e.tag.lower()})
            print("Explicit consolidation-tag local names:",names)
    except (requests.RequestException,ET.ParseError) as exc:
        print("UNVERIFIED:",exc)
else:
    print("Historical observed result: HTTP 200, 46,497 bytes, valid RDF/XML, 0 consolidation links found")

## 3. First SPARQL: how to avoid a silent false-negative
Do not equate a `200` and empty `results.bindings` with no current version. The early relation path and index search returned zero despite known consolidated CELEX values. The correction is to inspect raw CELEX results with a simple broad query.

In [ ]:
SUCCESS_QUERY = """
PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>
SELECT ?work ?celex WHERE {
  ?work cdm:resource_legal_id_celex ?celex .
  FILTER(CONTAINS(STR(?celex), "2019R1793"))
}
LIMIT 100
"""
print(SUCCESS_QUERY)
print("Key corrected predicate: cdm:resource_legal_id_celex")
print("Key corrected filter: FILTER(CONTAINS(STR(?celex), ...))")

## 4. Live SPARQL diagnostic with safe evidence capture
Shows original act, corrigenda and consolidated versions together. Parse only exact `^02019R1793-YYYYMMDD$` as consolidation candidates. A newest observed date does not prove the graph is complete or that a provision is effective.

In [ ]:
RUN_LIVE_SPARQL=True
record={"as_of_utc":datetime.now(timezone.utc).isoformat(),"celex":CELEX,"snapshot_date":SNAPSHOT,"status":"unverified","rows":None,"consolidations":[],"latest_observed":None,"comparison":"not_verified"}
if RUN_LIVE_SPARQL:
    try:
        r=requests.get(ENDPOINT,params={"query":SUCCESS_QUERY},headers={"Accept":"application/sparql-results+json"},timeout=45)
        record["http_status"]=r.status_code
        r.raise_for_status()
        data=r.json()
        rows=data["results"]["bindings"]
        record["rows"]=len(rows)
        if not rows:
            print("UNVERIFIED — query returned zero rows")
        else:
            candidates=[]
            for row in rows:
                celex=row.get("celex",{}).get("value","")
                match=re.fullmatch(r"02019R1793-(\d{8})",celex)
                if match:
                    try:
                        iso=datetime.strptime(match.group(1),"%Y%m%d").date().isoformat()
                        candidates.append({"celex":celex,"date":iso,"uri":row.get("work",{}).get("value")})
                    except ValueError:
                        pass
            record["consolidations"]=sorted(candidates,key=lambda x:x["date"],reverse=True)
            if record["consolidations"]:
                latest=record["consolidations"][0]
                record["latest_observed"]=latest["date"]
                record["status"]="metadata_found"
                record["comparison"]=("matches_latest_observed_consolidation" if latest["date"]==SNAPSHOT else "different_observed_date")
    except (requests.RequestException,ValueError,KeyError,TypeError) as exc:
        record["error"]=repr(exc)
out=DIAG_DIR/"cellar_sparql_diagnostic.json"
out.write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding="utf-8")
print("HTTP:",record.get("http_status"),"Rows:",record["rows"],"Status:",record["status"])
print("Latest observed:",record["latest_observed"],"Comparison:",record["comparison"])
print("Evidence:",out)

## 5. Offline regression checks (no network)
These assert only **parsing and selection rules**, not live currentness.

In [ ]:
def extract_consolidations(celex_values, basic="32019R1793"):
    pat=re.compile(r"^0"+re.escape(basic[1:])+r"-(\d{8})$")
    dates=[]
    for item in celex_values:
        m=pat.fullmatch(item)
        if m:
            try: dates.append(datetime.strptime(m.group(1),"%Y%m%d").date().isoformat())
            except ValueError: pass
    return sorted(set(dates))

example=["32019R1793","32019R1793R(01)","02019R1793-20250812","02019R1793-20260630","02023R0915-20260819"]
assert extract_consolidations(example)==["2025-08-12","2026-06-30"]
assert extract_consolidations([])==[]
print("PASS — distinguishes original act/corrigenda from matching consolidated CELEX IDs")

## 6. Postmortem and limits

**Root cause:** an incorrect assumption about Cellar data structures/search syntax, not an empty legal corpus. The HTTP 202 cause remains unconfirmed. The exact CELEX predicate plus a simple `FILTER(CONTAINS(...))` returned 19 records in the live diagnostic; NB06 v3 subsequently returned `metadata_found` for the 2019/1793 snapshot (2026-06-30).

**Operational rules:** validate connectivity → test simplest metadata query → inspect identifiers → parse exact consolidated CELEX → compare dates → keep `unverified` for no evidence. Never claim article-level change, entry into force, application dates, amendment completeness, or legal currentness from matching consolidated dates alone.

**Scope:** This supplementary notebook is for reproducing the debugging method. NB06 v3 remains the operative two-tool Q8 notebook.